<a href="https://colab.research.google.com/github/Raqeebir-Rab/2410-machine-learning/blob/main/RandomNodeGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf /content/drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install numpy networkx

In [3]:
import re

# ==========================================================
# Function: read_nodes()
# Purpose:
#   Reads an NS2 scene (.txt) file containing node coordinates.
#
# Input:
#   filename -> path of the scene file
#
# Output:
#   Dictionary in the following format:
#
#   {
#       node_id: {"X": value, "Y": value, "Z": value},
#       ...
#   }
#
# Example:
#   {
#       0: {"X": 10.5, "Y": 35.2, "Z": 0.0},
#       1: {"X": 18.9, "Y": 41.6, "Z": 0.0}
#   }
# ==========================================================
def read_nodes(filename):

    # Final dictionary containing all node coordinates
    nodes = {}

    # Regular expression used to extract:
    #   Node ID
    #   Coordinate type (X/Y/Z)
    #   Coordinate value
    #
    # Example matched line:
    # $node_(5) set X_ 125.45
    pattern = re.compile(r'\$node_\((\d+)\)\s+set\s+([XYZ])_\s+([-\d.]+)')

    # Temporary dictionary because X,Y,Z appear on separate lines
    temp = {}

    # Read scene file line by line
    with open(filename, "r") as f:

        for line in f:

            # Try matching the coordinate pattern
            m = pattern.search(line)

            # Ignore unrelated lines
            if not m:
                continue

            node_id = int(m.group(1))
            axis = m.group(2)          # X, Y or Z
            value = float(m.group(3))

            # Create dictionary entry if node is seen first time
            if node_id not in temp:
                temp[node_id] = {}

            # Store coordinate
            temp[node_id][axis] = value

    # Ensure every node contains X,Y,Z coordinates
    # Missing coordinates are replaced with 0.0
    for node_id in temp:

        nodes[node_id] = {
            "X": temp[node_id].get("X", 0.0),
            "Y": temp[node_id].get("Y", 0.0),
            "Z": temp[node_id].get("Z", 0.0)
        }

    return nodes


# ==========================================================
# Function: write_scene()
#
# Purpose:
#   Writes the updated node coordinates into a new NS2 scene file.
#
# Input:
#   filename -> output file
#   nodes    -> dictionary containing node coordinates
#
# The output preserves the NS2 scene format.
# ==========================================================
def write_scene(filename, nodes):

    with open(filename, "w") as f:

        # Write header
        f.write(f"#\n# nodes: {len(nodes)}\n#\n")

        # Write nodes in ascending order
        for node_id in sorted(nodes.keys()):

            x = nodes[node_id]["X"]
            y = nodes[node_id]["Y"]
            z = nodes[node_id]["Z"]

            # Write coordinates with 6 decimal places
            f.write(f"$node_({node_id}) set X_ {x:.6f}\n")
            f.write(f"$node_({node_id}) set Y_ {y:.6f}\n")
            f.write(f"$node_({node_id}) set Z_ {z:.6f}\n")


# ==========================================================
# Function: swap_sink_with_last()
#
# Purpose:
#   Exchanges (swaps) the POSITION of the sink node with
#   the last node in the scene.
#
# Important:
#   The node IDs DO NOT change.
# This is useful when simulations assume the sink is the
# last node while the dataset stores it elsewhere.
# ==========================================================
def swap_sink_with_last(nodes, sink_node):

    # Get all node IDs in sorted order
    node_ids = sorted(nodes.keys())

    # Last node in the scene
    last_node = node_ids[-1]

    # Verify sink exists
    if sink_node not in nodes:
        raise ValueError(f"Sink node {sink_node} not found")

    print("Swapping positions:")
    print("Sink node:", sink_node)
    print("Last node:", last_node)

    # ------------------------------------------------------
    # Swap entire coordinate dictionaries
    #
    # Before:
    # sink_node -> {X,Y,Z}
    # last_node -> {X,Y,Z}
    #
    # After:
    # sink_node gets last_node coordinates
    # last_node gets sink coordinates
    # ------------------------------------------------------
    nodes[sink_node], nodes[last_node] = (
        nodes[last_node],
        nodes[sink_node]
    )

    return nodes


# ==========================================================
# Function: create_swapped_scene()
#
# Purpose:
#   Complete processing pipeline.
#
# Steps:
#     1. Read original scene.
#     2. Swap sink with last node.
#     3. Save updated scene.
# ==========================================================
def create_swapped_scene(input_file, output_file, sink_node):

    # Read all node coordinates
    nodes = read_nodes(input_file)

    # Swap sink position with last node
    nodes = swap_sink_with_last(nodes, sink_node)

    # Write updated coordinates
    write_scene(output_file, nodes)

    print("New swapped scene saved to:", output_file)


# ==========================================================
# Main Program
#
# Specify:
#   input scene
#   output scene
#   sink node ID
#
# The program creates a new scene where the sink location
# is exchanged with the last node's location.
# ==========================================================

input_file = "/content/drive/MyDrive/Dataset/newRandom/Scene_18/scene-18.txt"

output_file = "/content/drive/MyDrive/Dataset/newRandom/scene-18.txt"

# Current sink node ID
sink_node = 14

# Execute the complete workflow
create_swapped_scene(input_file, output_file, sink_node)

Swapping positions:
Sink node: 14
Last node: 18
New swapped scene saved to: /content/drive/MyDrive/Dataset/newRandom/scene-18.txt


In [4]:
import numpy as np
import networkx as nx
import re


# ==========================================================
# Function: read_nodes()
#
# Purpose:
#   Reads an NS2 scene file and extracts the X and Y
#   coordinates of every node.
#
# Input:
#   filename -> scene (.txt) file
#
# Output:
#   Dictionary:
#
#   {
#       node_id : (X, Y),
#       ...
#   }
#
# Note:
#   The Z coordinate is ignored because clustering is
#   performed in a 2D deployment area.
# ==========================================================
def read_nodes(filename):

    nodes = {}

    # Regular expression for extracting:
    # Node ID
    # Coordinate type (X,Y,Z)
    # Coordinate value
    pattern = re.compile(r'\$node_\((\d+)\)\s+set\s+([XYZ])_\s+([-\d.]+)')

    temp = {}

    with open(filename, "r") as f:

        for line in f:

            m = pattern.search(line)

            if not m:
                continue

            node_id = int(m.group(1))
            axis = m.group(2)
            value = float(m.group(3))

            if node_id not in temp:
                temp[node_id] = {}

            temp[node_id][axis] = value

    # Store only X and Y coordinates
    for node_id in temp:

        nodes[node_id] = (
            temp[node_id].get("X", 0.0),
            temp[node_id].get("Y", 0.0)
        )

    return nodes


# ==========================================================
# Function: valid_distance()
#
# Purpose:
#   Determines whether a point can communicate with another
#   point according to the communication constraints.
#
# Rules:
#
# 1. Horizontal or vertical communication
#    (same X or same Y coordinate)
#       Maximum distance = 100
#
# 2. Diagonal communication
#       Maximum distance = 150
#
# Returns:
#   True  -> valid communication
#   False -> communication not allowed
# ==========================================================
def valid_distance(p1, p2):

    dx = abs(p1[0] - p2[0])
    dy = abs(p1[1] - p2[1])

    # Euclidean distance
    dist = np.linalg.norm(p1 - p2)

    # Horizontal or vertical neighbor
    if dx == 0 or dy == 0:
        return dist <= 100

    # Diagonal neighbor
    else:
        return dist <= 150


# ==========================================================
# Function: min_cost_balanced_kmeans()
#
# Purpose:
#   Performs balanced K-Means clustering using a
#   Minimum-Cost Flow optimization.
#
# Unlike standard K-Means:
#
#   • Every cluster receives almost equal number of nodes.
#   • Invalid assignments are discouraged using penalties.
#
# Parameters:
#
# points
#     Coordinates of all sensor nodes.
#
# k
#     Number of clusters to create.
#
# max_iter
#     Number of K-Means iterations.
#
# Returns:
#
# labels
#     Cluster ID assigned to every node.
# ==========================================================
def min_cost_balanced_kmeans(points, k, max_iter=10):

    n = len(points)

    # ------------------------------------------------------
    # Compute balanced cluster sizes.
    #
    # Example:
    #
    # n = 300 nodes
    # k = 10 clusters
    #
    # Each cluster gets 30 nodes.
    #
    # If not perfectly divisible, the first few clusters
    # receive one additional node.
    # ------------------------------------------------------
    base = n // k
    remainder = n % k

    cluster_sizes = [
        base + 1 if i < remainder else base
        for i in range(k)
    ]

    # Randomly initialize cluster centroids
    centroids = points[np.random.choice(n, k, replace=False)]

    # Large penalty discouraging invalid assignments
    PENALTY = 10**6

    # Repeat K-Means iterations
    for _ in range(max_iter):

        # Directed graph used for Minimum Cost Flow
        G = nx.DiGraph()

        source = "s"
        sink = "t"

        # ----------------------------------------------
        # Source → Sensor nodes
        #
        # Every node supplies one unit of flow.
        # ----------------------------------------------
        for i in range(n):
            G.add_edge(
                source,
                f"node_{i}",
                capacity=1,
                weight=0
            )

        # ----------------------------------------------
        # Sensor node → Cluster
        #
        # Every node may connect to every cluster.
        #
        # Cost:
        #   Euclidean distance
        #
        # Invalid assignments receive a very large penalty.
        # ----------------------------------------------
        for i in range(n):

            for j in range(k):

                dist = np.linalg.norm(
                    points[i] - centroids[j]
                )

                if valid_distance(points[i], centroids[j]):
                    weight = int(dist * 1000)
                else:
                    weight = int(dist * 1000) + PENALTY

                G.add_edge(
                    f"node_{i}",
                    f"cluster_{j}",
                    capacity=1,
                    weight=weight
                )

        # ----------------------------------------------
        # Cluster → Sink
        #
        # Capacity equals desired cluster size,
        # guaranteeing balanced clusters.
        # ----------------------------------------------
        for j in range(k):

            G.add_edge(
                f"cluster_{j}",
                sink,
                capacity=cluster_sizes[j],
                weight=0
            )

        # Solve Minimum Cost Flow
        flow = nx.max_flow_min_cost(
            G,
            source,
            sink
        )

        labels = np.zeros(n, dtype=int)

        # Recover assigned cluster for every node
        for i in range(n):

            for j in range(k):

                if (
                    f"cluster_{j}" in flow[f"node_{i}"]
                    and flow[f"node_{i}"][f"cluster_{j}"] > 0
                ):
                    labels[i] = j
                    break

        # ----------------------------------------------
        # Update centroids using current assignments
        # ----------------------------------------------
        for j in range(k):

            pts = points[labels == j]

            if len(pts) > 0:
                centroids[j] = pts.mean(axis=0)

    return labels


# ==========================================================
# Function: get_heads()
#
# Purpose:
#   Selects one Cluster Head (CH) per cluster.
#
# Method:
#   Compute cluster centroid.
#
#   Choose the node nearest to the centroid.
#
# Returns:
#   Dictionary:
#
#   {
#      cluster : head_node_index
#   }
# ==========================================================
def get_heads(points, labels, k):

    heads = {}

    for j in range(k):

        idxs = np.where(labels == j)[0]

        centroid = points[idxs].mean(axis=0)

        head = min(
            idxs,
            key=lambda x:
            np.linalg.norm(points[x] - centroid)
        )

        heads[j] = head

    return heads


# ==========================================================
# Function: assign_sink_to_cluster()
#
# Purpose:
#   Determines which cluster should contain the sink.
#
# Method:
#
#   1. Search all nodes.
#   2. Consider only nodes satisfying communication range.
#   3. Choose nearest valid node.
#   4. Assign sink to that node's cluster.
#
# If no valid node exists,
# assign to nearest cluster regardless of distance.
# ==========================================================
def assign_sink_to_cluster(
        sink_point,
        points,
        labels):

    best_cluster = None
    best_dist = float("inf")

    # Search valid neighbors
    for i, p in enumerate(points):

        if valid_distance(sink_point, p):

            dist = np.linalg.norm(
                sink_point - p
            )

            if dist < best_dist:
                best_dist = dist
                best_cluster = labels[i]

    # Fallback
    if best_cluster is None:

        for i, p in enumerate(points):

            dist = np.linalg.norm(
                sink_point - p
            )

            if dist < best_dist:
                best_dist = dist
                best_cluster = labels[i]

    return best_cluster


# ==========================================================
# Function: run()
#
# Purpose:
#
# Complete clustering pipeline.
#
# Steps:
#
# 1. Read node coordinates.
# 2. Assume last node is sink.
# 3. Remove sink from clustering.
# 4. Generate balanced clusters.
# 5. Select Cluster Heads.
# 6. Assign sink to nearest cluster.
# 7. Save cluster information.
#
# Output format:
#
# H node cluster
#     Cluster Head
#
# M node cluster
#     Cluster Member
#
# S node cluster
#     Sink
# ==========================================================
def run(input_file, output_file, k=2):

    coords = read_nodes(input_file)

    node_ids = sorted(coords.keys())

    # Last node is assumed to be sink
    sink_node = node_ids[-1]

    print("Sink node (last):", sink_node)

    # Remove sink before clustering
    cluster_nodes = node_ids[:-1]

    points = np.array(
        [coords[i] for i in cluster_nodes]
    )

    # Generate balanced clusters
    labels = min_cost_balanced_kmeans(
        points,
        k
    )

    # Select Cluster Heads
    heads = get_heads(
        points,
        labels,
        k
    )

    # Assign sink
    sink_point = np.array(coords[sink_node])

    sink_cluster = assign_sink_to_cluster(
        sink_point,
        points,
        labels
    )

    # Save cluster information
    with open(output_file, "w") as f:

        for idx, node in enumerate(cluster_nodes):

            cluster = labels[idx] + 1

            if idx == heads[labels[idx]]:
                f.write(f"H {node} {cluster}\n")
            else:
                f.write(f"M {node} {cluster}\n")

        # Write sink information
        f.write(
            f"S {sink_node} {sink_cluster + 1}\n"
        )

    print("Final clustering saved to:", output_file)


# ==========================================================
# Main Program
#
# Specify:
#   Input scene
#   Output cluster information
#   Number of clusters (k)
# ==========================================================

input_file = "/content/drive/MyDrive/Dataset/newRandom/scene-18.txt"

output_file = "/content/drive/MyDrive/Dataset/newRandom/clusterInfo.txt"

# Generate 150 balanced clusters
run(input_file, output_file, k=2)

Sink node (last): 18
Final clustering saved to: /content/drive/MyDrive/Dataset/newRandom/clusterInfo.txt


In [5]:
import re
import math
from collections import defaultdict
import os


# ==========================================================
# Function: read_node_positions()
#
# Purpose:
#   Reads an NS2 scene file and extracts the X and Y
#   coordinates of every node.
#
# Input:
#   filename -> path to the scene file
#
# Output:
#   Dictionary:
#
#   {
#       node_id : (X, Y),
#       ...
#   }
#
# Only X and Y coordinates are required because the
# traffic generation is based on node locations in a
# two-dimensional deployment area.
# ==========================================================
def read_node_positions(filename):

    nodes = {}

    # Regular expression used to extract:
    #   Node ID
    #   Coordinate type (X or Y)
    #   Coordinate value
    pattern = re.compile(r"\$node_\((\d+)\)\s+set\s+([XY])_\s+([-\d.]+)")

    with open(filename, "r") as f:

        for line in f:

            m = pattern.search(line)

            if not m:
                continue

            node_id = int(m.group(1))
            axis = m.group(2)
            value = float(m.group(3))

            if node_id not in nodes:
                nodes[node_id] = {"X": 0.0, "Y": 0.0}

            nodes[node_id][axis] = value

    # Convert dictionary into:
    # node_id -> (X,Y)
    return {k: (v["X"], v["Y"]) for k, v in nodes.items()}


# ==========================================================
# Function: read_cluster_info()
#
# Purpose:
#   Reads the cluster information file generated by the
#   clustering algorithm.
#
# Input file format:
#
#   H 12 3
#   M 13 3
#   M 14 3
#   ...
#   S 300 5
#
# H = Cluster Head
# M = Cluster Member
# S = Sink
#
# Returns:
#
# cluster_map
#     Maps every node to its cluster ID.
#
# cluster_heads
#     Set containing all Cluster Head nodes.
# ==========================================================
def read_cluster_info(filename):

    cluster_map = {}

    cluster_heads = set()

    with open(filename, "r") as f:

        for line in f:

            parts = line.split()

            if len(parts) < 3:
                continue

            level = parts[0]

            node = int(parts[1])

            cluster = int(parts[2])

            cluster_map[node] = cluster

            if level == "H":
                cluster_heads.add(node)

    return cluster_map, cluster_heads


# ==========================================================
# Function: distance()
#
# Purpose:
#   Computes Euclidean distance between two nodes.
#
# Formula:
#
#      √((x2-x1)^2 + (y2-y1)^2)
# ==========================================================
def distance(p1, p2):

    return math.sqrt(
        (p1[0]-p2[0])**2 +
        (p1[1]-p2[1])**2
    )


# ==========================================================
# Function: get_sink_neighbors()
#
# Purpose:
#   Finds all nodes located within a specified communication
#   range of the sink node.
#
# Default threshold:
#       150 meters
#
# These nodes can optionally be excluded from traffic
# generation because they communicate directly with the sink.
# ==========================================================
def get_sink_neighbors(
        nodes_pos,
        sink,
        threshold=150):

    neighbors = set()

    for node, pos in nodes_pos.items():

        if node == sink:
            continue

        if distance(pos, nodes_pos[sink]) <= threshold:
            neighbors.add(node)

    return neighbors


# ==========================================================
# Function: generate_traffic()
#
# Purpose:
#
# Generates a traffic configuration file for the simulator.
#
# Workflow:
#
# 1. Read node coordinates.
# 2. Read cluster assignments.
# 3. Identify sink node.
# 4. Find neighboring nodes of sink.
# 5. Exclude sink and Cluster Heads.
# 6. Select traffic-generating nodes.
# 7. Assign traffic types.
# 8. Save traffic file.
# ==========================================================
def generate_traffic(
        scene_file,
        cluster_file,
        output_file):

    # Read node coordinates
    nodes_pos = read_node_positions(scene_file)

    # Read clustering information
    cluster_map, cluster_heads = read_cluster_info(
        cluster_file
    )

    node_ids = sorted(nodes_pos.keys())

    # Last node is assumed to be sink
    sink = node_ids[-1]

    # Nodes within communication range of sink
    sink_neighbors = get_sink_neighbors(
        nodes_pos,
        sink
    )

    # ------------------------------------------------------
    # Exclude nodes that should not generate traffic
    #
    # Excluded:
    #   Sink
    #   Cluster Heads
    #
    # Sink neighbors can also be excluded if desired.
    # ------------------------------------------------------
    excluded = set([sink])

    excluded.update(cluster_heads)

    # Uncomment if sink neighbors should also be excluded
    # excluded.update(sink_neighbors)

    print("Sink:", sink)

    print("Cluster Heads:", cluster_heads)

    print("Sink Neighbors:", sink_neighbors)

    # ------------------------------------------------------
    # Group nodes by cluster
    #
    # Only non-excluded nodes become traffic generators.
    # ------------------------------------------------------
    clusters = defaultdict(list)

    for node, cluster in cluster_map.items():

        if node not in excluded:

            clusters[cluster].append(node)

    traffic_lines = []

    # ------------------------------------------------------
    # Assign traffic inside each cluster
    #
    # Requirement:
    #
    # Each cluster must contain at least 4 usable nodes.
    #
    # Assignment:
    #
    # First two nodes
    #     -> Type A traffic
    #
    # Next two nodes
    #     -> Type B traffic
    # ------------------------------------------------------
    for cluster_id, nodes in clusters.items():

        nodes = sorted(nodes)

        if len(nodes) < 4:

            print(
                f"Cluster {cluster_id} "
                f"has only {len(nodes)} usable nodes "
                f"(needs 4)"
            )

            continue

        # First two nodes generate Type A traffic
        a_nodes = nodes[:2]

        # Next two nodes generate Type B traffic
        b_nodes = nodes[2:4]

        # Generate traffic entries
        for n in a_nodes:

            traffic_lines.append(
                f"self a {n} 5"
            )

        for n in b_nodes:

            traffic_lines.append(
                f"self b {n} 5"
            )

    # ------------------------------------------------------
    # Write simulator traffic configuration
    # ------------------------------------------------------
    with open(output_file, "w") as f:

        # Simulation parameters
        f.write("avail 5000\n")
        f.write("p0 1\n")
        f.write("p1 2\n")

        # Communication ranges
        f.write("p0_range 100\n")
        f.write("p1_range 150\n")

        # Receiver power level
        f.write("recvP 2\n")

        # Packet generation rates
        f.write("lambda_a 10\n")
        f.write("lambda_b 10\n")

        # Traffic entries
        for line in traffic_lines:
            f.write(line + "\n")

        # Sink already exists in topology
        f.write("sinkPresent 0\n")

    print("Traffic file saved to:", output_file)


# ==========================================================
# Function: process_all()
#
# Purpose:
#
# Automatically processes multiple scene files and
# corresponding cluster files.
#
# The function:
#
# 1. Matches scene files with cluster files.
# 2. Generates one traffic file for every pair.
#
#
# ==========================================================
def process_all(
        scene_folder,
        cluster_folder,
        output_folder):

    os.makedirs(
        output_folder,
        exist_ok=True
    )

    # Store every scene file
    scene_map = {}

    for f in os.listdir(scene_folder):

        if f.endswith(".txt"):

            base = f.replace(".txt", "")

            scene_map[base] = os.path.join(
                scene_folder,
                f
            )

    # Match cluster files with scenes
    for f in os.listdir(cluster_folder):

        if not f.endswith(".txt"):
            continue

        base_cluster = f.replace(
            "_cluster.txt",
            ""
        )

        if base_cluster not in scene_map:

            print(
                f"No matching scene for: {f}"
            )

            continue

        scene_file = scene_map[base_cluster]

        cluster_file = os.path.join(
            cluster_folder,
            f
        )

        output_file = os.path.join(
            output_folder,
            f"traffic_{base_cluster}.txt"
        )

        generate_traffic(
            scene_file,
            cluster_file,
            output_file
        )


# ==========================================================
# Main Program
#
# Specify:
#
#   Scene file
#   Cluster information file
#   Output traffic file
#
# Generates the simulator traffic configuration.
# ==========================================================

scene_file = "/content/drive/MyDrive/Dataset/newRandom/scene-18.txt"

cluster_file = "/content/drive/MyDrive/Dataset/newRandom/clusterInfo.txt"

output_file = "/content/drive/MyDrive/Dataset/newRandom/traffic_18_CH.txt"

generate_traffic(
    scene_file,
    cluster_file,
    output_file
)

Sink: 18
Cluster Heads: {10, 7}
Sink Neighbors: {7, 8, 9, 13, 15}
Traffic file saved to: /content/drive/MyDrive/Dataset/newRandom/traffic_18_CH.txt


In [6]:
import re
import os


# ==========================================================
# Function: read_cluster_info()
#
# Purpose:
#   Reads the cluster information file and extracts:
#
#   1. Cluster Head (CH) nodes
#   2. Sink node
#
# Expected input format:
#
#   H 12 1
#   M 13 1
#   M 14 1
#   ...
#   S 300 5
#
# H = Cluster Head
# M = Cluster Member
# S = Sink
#
# Returns:
#
#   cluster_heads
#       Sorted list containing all Cluster Head IDs.
#
#   sink
#       Sink node ID.
# ==========================================================
def read_cluster_info(filename):

    cluster_heads = []

    sink = None

    with open(filename, "r") as f:

        for line in f:

            parts = line.split()

            if len(parts) < 3:
                continue

            level = parts[0]

            node = int(parts[1])

            # Store Cluster Heads
            if level == "H":
                cluster_heads.append(node)

            # Store Sink
            elif level == "S":
                sink = node

    return sorted(cluster_heads), sink


# ==========================================================
# Function: generate_fixed_traffic()
#
# Purpose:
#
# Generates a traffic configuration file in which
# only Cluster Heads generate traffic.
#
# Traffic Assignment:
#
# Cluster Heads are assigned traffic alternately:
#
# CH1 → Type A
# CH2 → Type B
# CH3 → Type A
# CH4 → Type B
# ...
#
# Each Cluster Head generates the same number
# of packets.
#
# The total number of Type A and Type B packets
# is used to automatically compute lambda values.
# ==========================================================
def generate_fixed_traffic(
        cluster_file,
        output_file):

    # Read Cluster Heads and sink node
    cluster_heads, sink = read_cluster_info(
        cluster_file
    )

    # Verify that Cluster Heads exist
    if len(cluster_heads) == 0:
        raise ValueError(
            "No cluster heads found"
        )

    # ------------------------------------------------------
    # Number of packets generated by each Cluster Head
    # ------------------------------------------------------
    packets_per_ch = 20

    traffic_lines = []

    # Total packets of each traffic type
    count_a = 0
    count_b = 0

    # ------------------------------------------------------
    # Alternate traffic assignment
    #
    # Even-index Cluster Heads
    #       -> Type A
    #
    # Odd-index Cluster Heads
    #       -> Type B
    #
    # Example:
    #
    # CH1 → A
    # CH2 → B
    # CH3 → A
    # CH4 → B
    # ------------------------------------------------------
    for i, ch in enumerate(cluster_heads):

        if i % 2 == 0:

            traffic_lines.append(
                f"self a {ch} {packets_per_ch}"
            )

            count_a += packets_per_ch

        else:

            traffic_lines.append(
                f"self b {ch} {packets_per_ch}"
            )

            count_b += packets_per_ch

    # ------------------------------------------------------
    # Total traffic generated by each application
    #
    # Lambda represents the total packet generation
    # rate for traffic Type A and Type B.
    # ------------------------------------------------------
    lambda_a = count_a

    lambda_b = count_b

    print("Cluster Heads:", cluster_heads)

    print(
        f"lambda_a={lambda_a}, "
        f"lambda_b={lambda_b}"
    )

    # ------------------------------------------------------
    # Write simulator traffic configuration
    # ------------------------------------------------------
    with open(output_file, "w") as f:

        # Simulation duration / availability
        f.write("avail 5000\n")

        # Packet priorities
        f.write("p0 1\n")
        f.write("p1 2\n")

        # Communication ranges
        f.write("p0_range 100\n")
        f.write("p1_range 150\n")

        # Receiver power level
        f.write("recvP 2\n")

        # Packet generation rates
        f.write(f"lambda_a {lambda_a}\n")
        f.write(f"lambda_b {lambda_b}\n")

        # Traffic assignments
        for line in traffic_lines:
            f.write(line + "\n")

        # Indicates that the sink is already present
        # in the network topology
        f.write("sinkPresent 1\n")

    print("Traffic file saved:", output_file)


# ==========================================================
# Function: process_all()
#
# Purpose:
#
# Processes every cluster information file contained
# in a folder and automatically generates one
# traffic configuration file for each network.
#
# Output:
#
# traffic_scene-1.txt
# traffic_scene-2.txt
# ...
# ==========================================================
def process_all(
        cluster_folder,
        output_folder):

    # Create output directory if it does not exist
    os.makedirs(
        output_folder,
        exist_ok=True
    )

    # Process every cluster file
    for f in os.listdir(cluster_folder):

        if not f.endswith(".txt"):
            continue

        # Remove "_cluster" suffix
        base_cluster = f.replace(
            "_cluster.txt",
            ""
        )

        cluster_file = os.path.join(
            cluster_folder,
            f
        )

        output_file = os.path.join(
            output_folder,
            f"traffic_{base_cluster}.txt"
        )

        generate_fixed_traffic(
            cluster_file,
            output_file
        )


# ==========================================================
# Main Program
#
# Specify:
#
#   Cluster information file
#   Output traffic configuration file
#
# Generates a simulator traffic file in which only
# Cluster Heads produce traffic.
# ==========================================================

cluster_file = "/content/drive/MyDrive/Dataset/newRandom/clusterInfo.txt"

output_file = "/content/drive/MyDrive/Dataset/newRandom/traffic-18.txt"

generate_fixed_traffic(
    cluster_file,
    output_file
)

Cluster Heads: [7, 10]
lambda_a=20, lambda_b=20
Traffic file saved: /content/drive/MyDrive/Dataset/newRandom/traffic-18.txt


In [8]:
import os


# ==========================================================
# Function: clean_name()
#
# Purpose:
#   Cleans file names when processing multiple files.
#
# Removes:
#   • "Copy of "
#   • "_cluster"
#
# Example:
#
#   Copy of scene-18_cluster.txt
#
# becomes
#
#   scene-18.txt
#
# This creates cleaner output filenames.
# ==========================================================
def clean_name(filename):

    name = filename.replace("Copy of ", "")

    name = name.replace("_cluster", "")

    return name


# ==========================================================
# Function: generate_sink_file()
#
# Purpose:
#
# Generates a simplified cluster information file
# in which:
#
#   • Every node belongs to a single cluster.
#   • The last node becomes the Cluster Head
#     (representing the sink).
#
# Input:
#
# Original cluster file:
#
# H 5 1
# M 8 1
# M 12 2
# S 30 4
#
# Output:
#
# M 5 1
# M 8 1
# M 12 1
# H 30 1
#
# The resulting file contains only one cluster.
# ==========================================================
def generate_sink_file(
        input_file,
        output_file):

    nodes = []

    # ------------------------------------------------------
    # Read every node ID from the input file.
    #
    # Ignore whether it is:
    #
    # H = Cluster Head
    # M = Member
    # S = Sink
    #
    # Only the node ID is required.
    # ------------------------------------------------------
    with open(input_file, "r") as f:

        for line in f:

            parts = line.split()

            if len(parts) < 2:
                continue

            node = int(parts[1])

            nodes.append(node)

    # ------------------------------------------------------
    # Sort node IDs.
    #
    # The highest node ID is assumed to be the sink.
    # ------------------------------------------------------
    nodes.sort()

    # ------------------------------------------------------
    # Create new cluster file.
    #
    # Every node belongs to Cluster 1.
    #
    # The final node (largest ID) becomes
    # the Cluster Head.
    # ------------------------------------------------------
    with open(output_file, "w") as f:

        for i, node in enumerate(nodes):

            # Last node
            if i == len(nodes) - 1:

                # Cluster Head (Sink)
                f.write(f"H {node} 1\n")

            else:

                # Cluster Member
                f.write(f"M {node} 1\n")


# ==========================================================
# Function: process_folder()
#
# Purpose:
#
# Automatically converts every cluster information
# file inside a folder into the simplified
# single-cluster format.
#
# Workflow:
#
# 1. Read every cluster file.
# 2. Clean the filename.
# 3. Generate a new sink file.
# 4. Save to output folder.
# ==========================================================
def process_folder(
        cluster_folder,
        output_folder):

    # Create output directory if it does not exist
    os.makedirs(
        output_folder,
        exist_ok=True
    )

    # Process every cluster file
    for file in os.listdir(cluster_folder):

        if not file.endswith(".txt"):
            continue

        input_path = os.path.join(
            cluster_folder,
            file
        )

        # Create cleaned filename
        output_name = clean_name(file)

        output_path = os.path.join(
            output_folder,
            output_name
        )

        # Generate simplified sink file
        generate_sink_file(
            input_path,
            output_path
        )

        print(f"{file}  →  {output_name}")


# ==========================================================
# Main Program
#
# Specify:
#
#   Input cluster file
#   Output sink file
#
# The generated file contains a single cluster
# where the last node acts as the Cluster Head
# (sink).
# ==========================================================

cluster_file = "/content/drive/MyDrive/Dataset/newRandom/clusterInfo.txt"

output_file = "/content/drive/MyDrive/Dataset/newRandom/sinkInfo.txt"

generate_sink_file(
    cluster_file,
    output_file
)